# v4-C finale: training su tutti i casi etichettati (2015-2022), predizioni per i 725 casi Kaggle (2023-01-01 -> 2024-12-25)
Stesso modello di v4-C (nessun tuning: non ha migliorato). Cambia solo la split: il train e' tutto il dataset etichettato (2922 casi), il test sono i casi di `prediction_mapping.csv`
(non hanno etichette: la metrica la calcola Kaggle). Il file va in formato **Run No + 2352 colonne** con i nomi originali del CSV: controlla che coincida con il sample submission.

In [1]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import joblib
from functools import cached_property
from pathlib import Path
from numpy.lib.stride_tricks import sliding_window_view as swv
import yaml
import xgboost as xgb

In [2]:
# create the classes and funcs to prepare the data
def build_cases(X, Y, T=168):
    '''
    Modify the dataset to have them case indexed
    Input:
    - X: x dataset
    - Y: y dataset
    '''
    
    # splitting between daily and hourly features
    hourly = ['day_ahead_price'] + [c for c in X.columns if c.endswith('_inflow')]
    vol = [c for c in X.columns if c.endswith('_volume')]
    wv  = [c for c in X.columns if c.endswith('_wv')]

    # getting the position of every case starting hour
    pos = X.index.get_indexer(Y.index)
    assert (pos >= 0).all() and (pos + T < len(X)).all()

    # creating the sliding windows
    win = swv(X[hourly].to_numpy(np.float32), T, axis=0)       
    xh = win[pos].reshape(len(pos), -1)                        
    
    wv_arr = X[wv].to_numpy()
    xs = np.hstack([
        X[vol].to_numpy()[pos],  # initial volume (s)
        wv_arr[pos],             # initial water value (pos)
        wv_arr[pos + T]          # final water value (pos+T)
    ])

    # getting the names for the columns of x
    hourly_cols = [f"{c}_t{h}" for c in hourly for h in range(T)]
    wv_init_cols = [f"{c}_init" for c in wv]
    wv_final_cols = [f"{c}_final" for c in wv]
    
    all_cols = hourly_cols + vol + wv_init_cols + wv_final_cols

    # excluding run_no e total_calculation_time
    y = pd.DataFrame(
        Y.filter(like='result_committed').to_numpy(np.uint8),
        index=Y.index,
        columns=Y.columns[2:]
    )
    
    x = pd.DataFrame(
        np.hstack([xh, xs]).astype(np.float32),
        index=Y.index,
        columns=all_cols
    )
    
    return x, y


class DataLoader:
    # TODO: Change timestamp data from string to datatime object if available in the data source.
    # TODO: move this in the notebooks (or make it simpler/remove it)

    def __init__(self, data_source=None):
        self.data_source = Path(data_source) if data_source else data_dir

    def _csv(self, folder, filename, date_columns=()):
        file_path = self.data_source / folder / filename
        frame = pd.read_csv(file_path)
        for column in date_columns:
            if column in frame.columns:
                frame[column] = pd.to_datetime(frame[column])
        return frame

    def _yaml(self, folder, filename):
        file_path = self.data_source / folder / filename
        with file_path.open(encoding="utf-8") as stream:
            return yaml.safe_load(stream)

    @cached_property
    def day_ahead_price_data(self):
        return self._csv(
            "kernel",
            "Historical_day_ahead_price_2015_2025.csv",
            ("date",),
        )

    @cached_property
    def inflow_data(self):
        return self._csv("kernel", "Historical_inflow_1958_2025.csv", ("date",))

    @cached_property
    def volume_data(self):
        return self._csv("kernel", "Historical_volume_2015_2024.csv", ("date",))

    @cached_property
    def synthetic_water_value_data(self):
        return self._csv(
            "kernel",
            "Synthetic_water_value_2015_2024.csv",
            ("date",),
        )

    @cached_property
    def commitment_descision_data(self):
        return self._csv("kernel", "Unit_commitment_decisions.csv", ("starttime",))

    @cached_property
    def commitment_decision_data(self):
        return self.commitment_descision_data

    @cached_property
    def min_flow_constraint_data(self):
        return self._csv("extended", "Constraint_min_flow.csv", ("date",))

    @cached_property
    def min_volume_constraint_data(self):
        return self._csv("extended", "Constraint_min_volume.csv", ("date",))

    @cached_property
    def topology_data(self): #TODO: CHECK THIS ONE LATER
        return self._yaml("extended", "Tokke_Vinje_topology.yaml")



# Class to create the entire dataset in one go
class Training_set(DataLoader):
    '''
    It imports all the dataset, merging them using the the index (that will be the date)
    '''
 
    def __init__(self, data_source=None):
        super().__init__(data_source)
        self.y = None
        self.x = None
 
    def load_copies(self):
        sources = {
            "day_ahead_price": self.day_ahead_price_data,
            "inflow": self.inflow_data,
            "volume": self.volume_data,
            "wv": self.synthetic_water_value_data,
            "uc": self.commitment_decision_data,
        }
        return sources
 
 
    def create_x(self):
        '''
        Create the complete X dataset
        
        TODO: feature engineering and selection
                + time split for easier computation (test model on smaller data)
        '''
        
        data = self.load_copies()
        
        # setting the date as index and fixing the columns names
        for name in data.keys():
            if 'date' in list(data[name].columns):
                data[name]['date'] = pd.to_datetime(data[name]['date'], utc=True)
                data[name] = data[name].set_index(data[name]['date']).drop(columns='date')
                if 'Day-ahead price (EUR/MWh)' in data[name].columns:
                    data[name].columns = data[name].columns.str.replace(r'\s*\([^)]*\)', '', regex=True).str.replace(r'[\s-]+', '_', regex=True).str.lower()
                else:
                    data[name].columns = data[name].columns.str.lower().str.replace(' ', '_')
                
            else:
                data[name]['starttime'] = pd.to_datetime(data[name]['starttime'], utc=True)
                data[name] = data[name].set_index(data[name]['starttime']).drop(columns='starttime')
                data[name].columns = data[name].columns.str.lower().str.replace(' ', '_')

        # wv and volume have daily data
        #   need to repet thier value hourly
        hourly = data['day_ahead_price'].index
        for name in ('volume', 'wv'):
            data[name] = data[name].reindex(hourly, method='ffill')
        
        # creating the two datset
        #   y stored for future use
        self.y = data['uc'].copy()
        data.pop('uc')
        x = data['day_ahead_price']
        data.pop('day_ahead_price')
        for name in data.keys():
                x = x.join(data[name].add_suffix('_' + name), how='left')
                
        
        self.x = x.copy()
    
    
    def split_small_data(self, begin_train:str ='2015-01-01', end_train: str='2018-12-31', begin_test: str='2019-01-12', end_test: str='2019-12-31'):
        '''
        Return X and Y dataset
        
        Input:
            - beginnig_train (str, dafault=2015-01-01 ): strart date training, format 'yyyy-mm-dd'
            - beginnig_test (str, dafault=2018-12-31 ): strart date testing, format 'yyyy-mm-dd'
            - end_train (str, dafault=2019-01-01 ): end date of training, format 'yyyy-mm-dd'
            - end_test (str, dafault=2019-12-31 ): end date of testing, format 'yyyy-mm-dd'
        '''
        
        self.create_x()
        X, Y = self.x.copy(), self.y.copy()
        x_train, y_train = build_cases(X, Y.loc[begin_train:end_train])
        x_test,  y_test  = build_cases(X, Y.loc[begin_test:end_test])
        return x_train, y_train, x_test, y_test

In [3]:
DATA = './Users/lucaag/data'                         # cartella dei dati (come nei notebook precedenti)
MAP = './Users/lucaag/data/prediction_mapping.csv'                       # Run No, starttime dei casi da predire
ts = Training_set(data_source=DATA)
ts.create_x()
X_all, Y_all = ts.x.sort_index(), ts.y.sort_index()
x_train, y_train = build_cases(X_all, Y_all)         # tutti i casi etichettati

mp = pd.read_csv(MAP)
mp['starttime'] = pd.to_datetime(mp['starttime'], utc=True)
Yd = pd.DataFrame(0, index=pd.DatetimeIndex(mp['starttime']), columns=Y_all.columns)   # etichette fittizie: servono solo a indicizzare i casi
x_test, y_test = build_cases(X_all, Yd)
topo = ts.topology_data['model']
print(x_train.shape, x_test.shape, '| colonne con NaN nel test:', int(x_test.isna().any().sum()))

(2922, 3243) (725, 3243) | colonne con NaN nel test: 0


In [4]:
T = 168
RES = {k.lower(): v for k, v in topo['reservoir'].items()}
PL = {k.lower().replace(' ', '_'): v for k, v in topo['plant'].items()}

# centrale: (bacini a monte, bacino a valle), da Tokke_Vinje_topology.yaml
TOPO = {
    'vesle_kjela': (['kjelavatn'], 'foersvatn'),
    'kjela': (['foersvatn', 'bordalsvatn'], 'hyljelihyl'),
    'songa': (['songavatn', 'bitdalsvatn'], 'totak'),
    'vinje': (['vaamarvatn', 'totak'], 'vinjevatn'),
    'haukeli': (['vatjern', 'langeidvatn'], 'vinjevatn'),
    'tokke': (['vinjevatn'], 'bandak'),
    'byrte': (['botnedalsvatn'], 'byrtevatn'),
    'lio': (['byrtevatn'], 'bandak'),
    'hogga': (['bandak'], None),
}
UP_PLANTS = {'kjela': ['vesle_kjela'], 'vinje': ['kjela', 'songa'], 'tokke': ['vinje', 'haukeli'],
             'hogga': ['tokke', 'lio'], 'lio': ['byrte']}
ORDER = ['vesle_kjela', 'songa', 'byrte', 'haukeli', 'kjela', 'lio', 'vinje', 'tokke', 'hogga']
HAS_DOWN = {p for ps in UP_PLANTS.values() for p in ps}

units = sorted({c[len('result_committed_'):].rsplit('_t', 1)[0] for c in y_train.columns})
plant_units = {p: [i for i, u in enumerate(units) if u.rsplit('_g', 1)[0] == p] for p in ORDER}
plant_of = lambda u: units[u].rsplit('_g', 1)[0]
ucols = lambda u: [f'result_committed_{units[u]}_t{h}' for h in range(T)]
Y3 = np.stack([y_train[ucols(u)].to_numpy() for u in range(len(units))], axis=1)
Yflat = [Y3[:, u, :].ravel() for u in range(len(units))]
out_cols = [c for u in range(len(units)) for c in ucols(u)]

In [5]:
PRICE = [f'day_ahead_price_t{h}' for h in range(T)]

def shift(a, k):  # ritardo di k ore con riempimento ai bordi
    o = np.roll(a, k, 1)
    if k > 0: o[:, :k] = a[:, :1]
    else: o[:, k:] = a[:, -1:]
    return o

def head(r, v):
    d = RES[r]['vol_head']
    return np.interp(v, d['x'], d['y'])

def static(X, plant):  # feature per caso: volumi, water value, riempimento, salto netto
    up, dn = TOPO[plant]; rs = up + ([dn] if dn else [])
    S = [X[[f'{r}_{s}' for r in rs for s in ('volume', 'wv_init', 'wv_final')]].to_numpy()]
    S += [(X[f'{r}_volume'] / RES[r]['max_vol']).to_numpy()[:, None] for r in rs]
    hu = head(up[0], X[f'{up[0]}_volume'].to_numpy())
    hd = head(dn, X[f'{dn}_volume'].to_numpy()) if dn else np.zeros_like(hu)
    wd = X[f'{dn}_wv_final'].to_numpy() if dn else np.zeros(len(X))
    S.append(np.column_stack([hu, hu - np.maximum(hd, PL[plant]['outlet_line']),
                              X[f'{up[0]}_wv_final'].to_numpy() - wd]))
    return np.hstack(S)

def base(X, plant, up_prob=None):  # matrice (n*168, F), identica a v4
    up, n = TOPO[plant][0], len(X)
    P = X[PRICE].to_numpy(); D = P.reshape(n, 7, 24)
    I = sum((X[[f'{r}_inflow_t{h}' for h in range(T)]].to_numpy() for r in up if f'{r}_inflow_t0' in X.columns),
            np.zeros((n, T), np.float32))  # bacini senza inflow -> 0
    wvf = X[f'{up[0]}_wv_final'].to_numpy()[:, None]
    h = np.tile(np.arange(T), (n, 1))
    F = [h, h % 24, (X.index.dayofweek.to_numpy()[:, None] + h // 24) % 7,
         P, shift(P, 1), shift(P, -1), P.argsort(1).argsort(1) / (T - 1),
         D.argsort(2).argsort(2).reshape(n, T) / 23, P - np.repeat(D.mean(2), 24, 1),
         P - P.mean(1, keepdims=True), P - wvf, I, np.repeat(I.mean(1, keepdims=True), T, 1)]
    if up_prob is not None:  # probabilita' di accensione a monte (stacking)
        F += [up_prob, shift(up_prob, 1), shift(up_prob, 3), np.repeat(up_prob.mean(1, keepdims=True), T, 1)]
    A = np.column_stack([f.ravel() for f in F] + [np.repeat(static(X, plant), T, 0),
                                                  np.repeat(X.index.dayofyear.to_numpy(), T)])
    return A.astype(np.float32)

In [6]:
NE, FOLDS, GAP = 200, 5, 7
HP = {}  # iperparametri per centrale, riempito dal tuning; vuoto = v4

def mk(plant=None, **kw):
    p = dict(n_estimators=NE, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
             tree_method='hist', n_jobs=-1, random_state=0)
    return xgb.XGBClassifier(**{**p, **HP.get(plant, {}), **kw})

def fp(X, y, Xs, plant=None):  # fit + predict; target costanti -> probabilita' costante
    if y.min() == y.max():
        return [np.full(len(x), float(np.clip(y[0], .01, .99))) for x in Xs]
    m = mk(plant).fit(X, y)
    return [m.predict_proba(x)[:, 1] for x in Xs]

def folds(n):
    b = np.linspace(0, n, FOLDS + 1, dtype=int)
    for a, c in zip(b[:-1], b[1:]):
        yield np.r_[0:max(a - GAP, 0), min(c + GAP, n):n], np.arange(a, c)

rows = lambda c: (np.asarray(c)[:, None] * T + np.arange(T)).ravel()
design = lambda B, c, us: np.vstack([np.column_stack([B[rows(c)], np.full(len(rows(c)), u, np.float32)]) for u in us])
ylong = lambda c, us: np.concatenate([Yflat[u][rows(c)] for u in us])

def fit_group(B, Bt, us, oof):
    n, nt, pl = len(B) // T, len(Bt) // T, plant_of(us[0])
    allc = np.arange(n)
    te = fp(design(B, allc, us), ylong(allc, us), [design(Bt, np.arange(nt), us)], pl)[0]
    te = te.reshape(len(us), nt, T)
    oo = None
    if oof:
        oo = np.zeros((len(us), n, T))
        for tr, va in folds(n):
            p = fp(design(B, tr, us), ylong(tr, us), [design(B, va, us)], pl)[0]
            oo[:, va] = p.reshape(len(us), len(va), T)
    return oo, te

def run():  # variante C di v4: un modello per centrale + stacking; ritorna predizioni test e OOF per centrale
    oof, te, out = {}, {}, np.zeros((len(x_test), len(units), T), np.float32)
    for plant in ORDER:
        us, ups = plant_units[plant], UP_PLANTS.get(plant, [])
        up_tr = np.mean([oof[p] for p in ups], 0) if ups else None
        up_te = np.mean([te[p] for p in ups], 0) if ups else None
        B, Bt = base(x_train, plant, up_tr), base(x_test, plant, up_te)
        o, t = fit_group(B, Bt, us, plant in HAS_DOWN)
        out[:, us, :] = t.transpose(1, 0, 2)
        te[plant] = t.mean(0)
        if plant in HAS_DOWN: oof[plant] = o.mean(0)
        print(plant, 'ok')
    pred = pd.DataFrame(out.reshape(len(x_test), -1), index=y_test.index, columns=out_cols)
    return pred[y_test.columns], oof

auc = lambda pred: roc_auc_score(y_test.to_numpy(), pred.to_numpy(), average='micro')
per_unit = lambda pred: pd.Series({units[u]: roc_auc_score(y_test[ucols(u)].to_numpy().ravel(), pred[ucols(u)].to_numpy().ravel())
                                   for u in range(len(units)) if y_test[ucols(u)].to_numpy().min() != y_test[ucols(u)].to_numpy().max()})

## Training su tutto + predizione dei casi Kaggle

In [9]:
pred, _ = run()                                      # v4-C: un modello per centrale + stacking, HP di default
raw = ts.commitment_decision_data                    # colonne con i nomi originali (maiuscole, spazi)
names = dict(zip(Y_all.filter(like='result_committed').columns, raw.filter(like='result_committed').columns))
sub = pred.rename(columns=names)
sub.insert(0, 'Run No', mp['Run No'].to_numpy())
cols = pd.read_csv('./Users/lucaag/data/sample_submission.csv', nrows=0).columns      # stesso ordine e nomi del sample Kaggle
assert set(cols) == set(sub.columns), set(cols) ^ set(sub.columns)
sub = sub[cols]
assert (sub['Run No'].to_numpy() == pd.read_csv('./Users/lucaag/data/sample_submission.csv', usecols=['Run No'])['Run No'].to_numpy()).all()
sub.to_csv('./Users/lucaag/data/sample_submission.csv', index=False)
print(sub.shape, '| prob min/max:', float(sub.iloc[:, 1:].to_numpy().min()), float(sub.iloc[:, 1:].to_numpy().max()))
sub.iloc[:3, :4]

vesle_kjela ok
songa ok
byrte ok
haukeli ok
kjela ok
lio ok
vinje ok
tokke ok
hogga ok
(725, 2353) | prob min/max: 4.1737799620023e-05 0.9999591112136841


,Run No,result_committed_Hogga_G1_t0,result_committed_Hogga_G1_t1,result_committed_Hogga_G1_t2
starttime,,,,
2023-01-01 00:00:00+00:00,2923,0.095686,0.091894,0.082899
2023-01-02 00:00:00+00:00,2924,0.542342,0.528689,0.531579
2023-01-03 00:00:00+00:00,2925,0.991194,0.991162,0.986844


In [15]:
import os
print(os.getcwd())
out = '/mnt/batch/tasks/shared/LS_root/mounts/clusters/lucaag1/code/Users/lucaag/submission_v4c_all.csv'
sub.to_csv(out, index=False)
print(os.path.getsize(out) / 1e6, 'MB')

/mnt/batch/tasks/shared/LS_root/mounts/clusters/lucaag1/code
19.265062 MB
